# 1 · Core quality control and rock typing: Zeerijp-3A, Groningen Rotliegend

**Question.** Can a few rock types, defined from core porosity and permeability, predict permeability better than one porosity–permeability trend?

**Data.** Routine core analysis of well **ZEERIJP-03-SIDETRACK1 (ZRP-3A)**, operated by NAM and drilled in 2015 into the Rotliegend (Slochteren) sandstone of the Groningen field, published by the Dutch government on [NLOG](https://www.nlog.nl). 199.8 m of core in 8 runs. The notebook downloads the public file from NLOG and checks its SHA-256; the raw file is not copied into this repository.

**Method.** Quality control of the plugs, then the flow zone indicator (FZI) of Amaefule et al. (1993) and Gaussian-mixture rock types on log(FZI), with the number of types chosen by the Bayesian information criterion (BIC). This is the KAI-Subsurface rock typing workflow.

In [ ]:
import json, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio
import subsurface_tools as kt
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False, "savefig.bbox": "tight", "savefig.facecolor": "white"})
SAND, MUD, INK = "#d9a441", "#5b4237", "#1f2a36"
pathlib.Path("results").mkdir(exist_ok=True); pathlib.Path("figures").mkdir(exist_ok=True)
core_file = kt.fetch_nlog("core"); log_file = kt.fetch_nlog("logs"); sonic_file = kt.fetch_nlog("sonic")
plugs = kt.read_zrp_plugs(core_file)
print(len(plugs), "plug measurements from", plugs.sheet.nunique(), "sheets")

## Quality control
Plugs described as fractured, broken, cracked or disintegrated measure the damage, not the rock, and are set aside. Permeabilities reported as below the 0.01 mD detection limit are kept as a count but not in the fits.

In [ ]:
h = plugs[(plugs.orient == "H") & plugs.k_kl.notna() & (plugs.k_kl > 0) & plugs.phi.notna()]
qc = {"horizontal plugs with k": int(len(h)), "damaged (set aside)": int(h.damaged.sum()),
      "below 0.01 mD detection (all plugs)": int(plugs.k_below_detection.sum())}
h = h[~h.damaged].drop_duplicates("depth").sort_values("depth").copy()
qc["used"] = int(len(h)); print(qc)
print(h.groupby("core").size().rename("plugs per core").to_dict())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 6), sharey=True)
ax[0].plot(h.phi, h.depth, ".", ms=3, color=INK); ax[0].set_xlabel("porosity (-)"); ax[0].set_ylabel("depth (m along hole)"); ax[0].invert_yaxis()
ax[1].semilogx(h.k_kl, h.depth, ".", ms=3, color=SAND); ax[1].set_xlabel("Klinkenberg k (mD)")
v = plugs[(plugs.orient == "V") & plugs.k_kl.notna() & ~plugs.damaged]
ax[2].semilogx(v.k_kl, v.depth, "^", ms=4, color=MUD); ax[2].set_xlabel("vertical plug k (mD)")
fig.suptitle("ZRP-3A core: 8 cores, 3521-3867 m"); fig.tight_layout(); fig.savefig("figures/zrp_core_profile.png")

## Flow zone indicator and rock types

In [ ]:
h["RQI"], h["FZI"] = kt.fzi(h.phi, h.k_kl)
h["RT"], bic, n_rt = kt.rock_types(h.FZI, n=3)
_, bic, n_bic = kt.rock_types(h.FZI)
print("BIC by number of types:", {k: round(v, 1) for k, v in bic.items()}, "-> BIC choice", n_bic, "; used here:", n_rt)
fz = h.groupby("RT").FZI.apply(lambda s: float(np.exp(np.log(s).mean())))
print(pd.DataFrame({"plugs": h.RT.value_counts().sort_index(), "mean FZI (micron)": fz.round(2),
                    "median k (mD)": h.groupby("RT").k_kl.median().round(1), "median phi": h.groupby("RT").phi.median().round(3)}))

**Why three types and not the BIC choice.** BIC prefers two types; a third splits off the tight, cemented sandstones (27 plugs) that behave differently in flow. Both are shown below; the choice is a geological one, checked against the plug descriptions.

In [ ]:
lk = np.log10(h.k_kl)
a = np.polyfit(h.phi, lk, 1); single = np.polyval(a, h.phi)
typed = np.log10(kt.k_from_fzi(h.phi, h.RT.map(fz)))
res1 = {"one trend: within x3": round(kt.within(single, lk), 3), "three rock types: within x3": round(kt.within(typed, lk), 3),
        "one trend: rmse log10 k": round(float(np.sqrt(np.mean((single - lk) ** 2))), 3),
        "three rock types: rmse log10 k": round(float(np.sqrt(np.mean((typed - lk) ** 2))), 3)}
print(res1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
pp = np.linspace(.02, .34, 50)
RTC = {1: MUD, 2: "#b07a3b", 3: SAND}
for t in sorted(fz.index):
    s = h[h.RT == t]; ax[0].semilogy(s.phi, s.k_kl, "o", ms=3.5, alpha=.75, color=RTC[t], label=f"RT{t} (FZI {fz[t]:.2f})")
    ax[0].semilogy(pp, kt.k_from_fzi(pp, fz[t]), "-", lw=1.6, color=RTC[t])
ax[0].semilogy(pp, 10 ** np.polyval(a, pp), "k--", label="one trend"); ax[0].set_ylim(1e-3, 1e4)
ax[0].set_xlabel("porosity (-)"); ax[0].set_ylabel("k (mD)"); ax[0].legend(fontsize=8); ax[0].set_title("Rock types on the poro-perm plot")
ax[1].plot(lk, single, "o", ms=3, alpha=.45, color="#9aa7b4", label=f"one trend ({res1['one trend: within x3']:.0%} within x3)")
ax[1].plot(lk, typed, "o", ms=3, alpha=.6, color=SAND, label=f"rock types ({res1['three rock types: within x3']:.0%} within x3)")
ax[1].plot([-3, 4], [-3, 4], "k-", lw=.8); ax[1].set_xlabel("measured log10 k"); ax[1].set_ylabel("predicted log10 k"); ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig("figures/zrp_rock_types.png")

## Vertical against horizontal: what the plugs say about kv/kh

In [ ]:
v = plugs[(plugs.orient == "V") & plugs.k_kl.notna() & ~plugs.damaged].sort_values("depth")
pair = pd.merge_asof(v, h[["depth", "k_kl"]], on="depth", direction="nearest", tolerance=0.3, suffixes=("_v", "_h")).dropna(subset=["k_kl_h"])
pair["kv_kh"] = pair.k_kl_v / pair.k_kl_h
kvkh = {"pairs within 0.3 m": int(len(pair)), "P10": round(float(pair.kv_kh.quantile(.1)), 3),
        "P50": round(float(pair.kv_kh.median()), 3), "P90": round(float(pair.kv_kh.quantile(.9)), 3)}
print(kvkh)

At plug scale the sandstone is close to isotropic (median kv/kh about 1). That does **not** mean the reservoir is: the plugs are cut from sandstone, and the claystone laminae that control vertical flow are rarely plugged (9 of 10 claystone plugs measured below 0.01 mD). Notebook 3 takes this to the bed scale.

In [ ]:
h[["depth", "core", "phi", "k_kl", "FZI", "RT", "remark"]].to_csv("results/zrp3a_rock_types.csv", index=False)
json.dump({"qc": qc, "rock_types": {str(k): {"plugs": int((h.RT == k).sum()), "fzi": round(v, 3)} for k, v in fz.items()},
           "bic_choice": int(n_bic), "core_level": res1, "plug_kv_kh": kvkh}, open("results/nb1_rock_types.json", "w"), indent=1)
assert res1["three rock types: within x3"] > res1["one trend: within x3"]

## Result and limits
Three rock types reproduce the measured permeability within a factor of 3 for most plugs, clearly better than one trend. The rock types still need porosity **and** permeability, so they only exist where there is core. Notebook 2 asks how well they can be predicted from logs alone, in uncored intervals or wells.

*References:* Amaefule, J.O. et al. (1993) SPE 26436 · Data: NAM / NLOG, public under the Dutch Mining Act.